In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, utils
from torchvision.transforms import v2 as T
from tqdm import tqdm
from torchinfo import summary
import matplotlib.pyplot as plt

In [2]:
device = torch.device("mps")

In [ ]:
data_transforms = T.Compose([
    T.Resize((32, 32)),
    T.RandomRotation(30, interpolation=T.InterpolationMode.BILINEAR),
    T.RandomPerspective(distortion_scale=0.5, p=0.5, interpolation=T.InterpolationMode.BILINEAR),
    T.RandomResizedCrop((32, 32), scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    T.ToImage(),
    T.ToDtype(torch.bfloat16, scale=True),
])

train_dataset = datasets.QMNIST(root='../data', train=True, download=True, transform=data_transforms)
test_dataset = datasets.QMNIST(root='../data', train=False, download=True, transform=data_transforms)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

x, _ = next(iter(train_loader))
grid = utils.make_grid(x, nrow=16)
plt.figure(figsize=(16, 8))
plt.imshow(grid.permute(1, 2, 0))
plt.axis('off')
plt.title('Sample QMNIST Images')
plt.show()

In [ ]:
class Block(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, downsample: bool = True):
        super(Block, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.relu = nn.GELU()
        self.pool = nn.MaxPool2d(2) if downsample else nn.Identity()
    
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x)) + x
        x = self.pool(x)
        return x

class MNISTClassifier(nn.Module):
    def __init__(self):
        super(MNISTClassifier, self).__init__()
        self.backbone = nn.Sequential(
            Block(1, 16),
            Block(16, 32),
            Block(32, 32),
            Block(32, 32, downsample=False),
            Block(32, 32),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128,  10)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        classes = self.head(features)
        return classes
    
model = MNISTClassifier().to(device, dtype=torch.bfloat16)
summary(model, input_size=(1, 1, 32, 32), depth=2, device=device, dtypes=[torch.bfloat16])

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

n_epochs = 5
for epoch in range(n_epochs):
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs}")
    for x, y in pbar:

        out = model(x.to(device, dtype=torch.bfloat16))
        loss = criterion(out, y.to(device, dtype=torch.bfloat16))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        pbar.set_postfix({"loss": loss.item(), "acc": (out.argmax(dim=1) == y.to(device, dtype=torch.bfloat16)).float().mean().item(), "grad_norm": grad_norm.item()})

In [ ]:
# confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np

y_true, y_pred = [], []
with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Evaluating"):
        out = model(x.to(device, dtype=torch.bfloat16))
        preds = out.argmax(dim=1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print(f"Test Accuracy: {(np.array(y_true) == np.array(y_pred)).mean():.4f}")

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[str(i) for i in range(10)])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# classification report
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, digits=4))

In [ ]:
print(f"{'Name':<30} {'Grad Norm':<20} {'Weight Norm':<20} {'Ratio':<20}")
for name, param in model.named_parameters():
    if param.requires_grad:
        grad = param.grad.norm().item()
        weight = param.norm().item()
        ratio = grad / weight if weight > 0 else 0
        print(f"{name:<30} {grad:10.4f}            {weight:10.4f}     {ratio:10.4f}")